In [39]:
import pandas as pd
import pandas_ta as ta
import numpy as np
from statsmodels.tsa.stattools import adfuller

In [40]:
# 1. Load the Data from Day 1
# Point this to wherever your Parquet file was saved
df = pd.read_parquet('C:/Users/Sam Garcia/PycharmProjects/macro_alpha/src/data/market_macro_data.parquet')

print(f"Loaded {len(df)} rows. Starting Feature Engineering...")

Loaded 4040 rows. Starting Feature Engineering...


In [41]:
# ==========================================
# PART A: Technical Features (Market Psychology)
# ==========================================

# 1. RSI (Relative Strength Index) - 14 Day
# Measures overbought/oversold conditions (0 to 100)
df.ta.rsi(close='close_sp500', length=14, append=True)

# 2. MACD (Moving Average Convergence Divergence)
# Measures trend momentum. 'append=True' adds 3 columns: MACD line, Histogram, and Signal line.
df.ta.macd(close='close_sp500', fast=12, slow=26, signal=9, append=True)

# 3. Rolling Volatility (20-Day Standard Deviation of Returns)
# We first need daily returns, then we calculate the rolling standard deviation
df['daily_return'] = df['close_sp500'].pct_change()
df['volatility_20d'] = df['daily_return'].rolling(window=20).std() * np.sqrt(252) # Annualized

# Golden Cross / Death Cross signals
sma_50 = df['close_sp500'].rolling(50).mean()
sma_200 = df['close_sp500'].rolling(200).mean()

df['price_to_sma_50'] = df['close_sp500'] / sma_50
df['price_to_sma_200'] = df['close_sp500'] / sma_200

In [42]:
# ==========================================
# PART B: Macro Features (Economic Reality)
# ==========================================
# The absolute value of macro data matters less than the *velocity of change*.

# 1. Yield Curve Momentum
# Is the spread widening or flattening over the last month (21 trading days)?
df['yield_spread_1mo_change'] = df['yield_spread'].diff(21)

# 2. Fed Funds Rate Velocity
# Did the Fed raise or cut rates in the last 3 months (63 trading days)?
df['fed_funds_3mo_change'] = df['fed_funds_rate'].diff(63)

# 3. Inflation Momentum
# Is CPI accelerating or decelerating month-over-month?
# (We calculated 'inflation_mom' in Day 1, let's see its 3-month trend)
if 'inflation_mom' in df.columns:
    df['inflation_trend_3mo'] = df['inflation_mom'].diff(63)

# Macro indicators take 6-12 months to affect markets
df['fed_funds_6mo_lag'] = df['fed_funds_rate'].shift(126)  # 6 months = ~126 trading days

In [43]:
# ==========================================
# PART C: The Target Variable
# ==========================================
# Predict if the price 5 days from now will be HIGHER than today's price.

# Shift(-5) pulls the price from 5 days in the future to today's row
df['future_5d_close'] = df['close_sp500'].shift(-5)

# Target: 1 if Up, 0 if Down
df['target_5d_up'] = (df['future_5d_close'] > df['close_sp500']).astype(int)

In [44]:
# ==========================================
# Stationarity Check
# ==========================================

def check_stationarity(series, name):
    result = adfuller(series.dropna())
    print(f"{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Stationary: {'YES ✓' if result[1] < 0.05 else 'NO ✗'}\n")

# Test your features
check_stationarity(df['yield_10y'], 'Yield 10Y')
check_stationarity(df['yield_2y'], 'Yield 2Y')
check_stationarity(df['yield_spread'], 'Yield Spread')
check_stationarity(df['daily_return'], 'Daily Return')

Yield 10Y:
  ADF Statistic: -1.4247
  p-value: 0.5704
  Stationary: NO ✗

Yield 2Y:
  ADF Statistic: -0.8602
  p-value: 0.8008
  Stationary: NO ✗

Yield Spread:
  ADF Statistic: -1.9399
  p-value: 0.3135
  Stationary: NO ✗

Daily Return:
  ADF Statistic: -13.8051
  p-value: 0.0000
  Stationary: YES ✓



In [45]:
# ==========================================
# Convert into Stationary Features by first-differencing
# ==========================================

df['yield_10y_change'] = df['yield_10y'].diff()
df['yield_2y_change'] = df['yield_2y'].diff()
df['yield_spread_change'] = df['yield_spread'].diff()

In [46]:
# ==========================================
# PART D: Clean Up
# ==========================================
# Technical indicators and lags create NaNs at the beginning of the dataset.
# Shifting for the target creates NaNs at the end of the dataset.

# In training: drop last 5 rows (can't calculate target)
df_train = df[:-5].dropna()

# In production/inference: DON'T drop last rows
df_inference = df.dropna()  # Keep all rows where features exist

print(f"\nFinal dataset has {len(df_train)} rows and {len(df_train.columns)} columns.")
print("\nFeatures generated successfully:")
print(list(df_train.columns))

# Let's check the balance of our target
print("\nTraining Target Class Balance:")
print(df_train['target_5d_up'].value_counts(normalize=True))

print("\nInference Target Class Balance:")
print(df_inference['target_5d_up'].value_counts(normalize=True))


Final dataset has 3836 rows and 25 columns.

Features generated successfully:
['close_sp500', 'vix', 'yield_10y', 'yield_2y', 'fed_funds_rate', 'yield_spread', 'cpi', 'inflation_mom', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'daily_return', 'volatility_20d', 'price_to_sma_50', 'price_to_sma_200', 'yield_spread_1mo_change', 'fed_funds_3mo_change', 'inflation_trend_3mo', 'fed_funds_6mo_lag', 'future_5d_close', 'target_5d_up', 'yield_10y_change', 'yield_2y_change', 'yield_spread_change']

Training Target Class Balance:
target_5d_up
1    0.605579
0    0.394421
Name: proportion, dtype: float64

Inference Target Class Balance:
target_5d_up
1    0.605579
0    0.394421
Name: proportion, dtype: float64


In [47]:
# Drop columns that cause Data Leakage or Non-Stationarity
columns_to_drop = [
    'close_sp500',      # Non-stationary raw price
    'cpi',              # Non-stationary raw index
    'future_5d_close',   # Data leakage (The Future)
    'yield_10y',         # Non-stationary raw yield
    'yield_2y',          # Non-stationary raw yield
    'yield_spread',      # Non-stationary raw yield spread
]

df_train_ready = df_train.drop(columns=columns_to_drop)
df_inference_ready = df_inference.drop(columns=columns_to_drop)

# Save the final engineered dataset for the ML Models
df_train_ready.to_parquet('C:/Users/Sam Garcia/PycharmProjects/macro_alpha/data/processed/train_ready_features.parquet')
df_inference_ready.to_parquet('C:/Users/Sam Garcia/PycharmProjects/macro_alpha/data/processed/inference_ready_features.parquet')

print("✅ Ready for Machine Learning. Final Features:")
print(list(df_train_ready.columns))

✅ Ready for Machine Learning. Final Features:
['vix', 'fed_funds_rate', 'inflation_mom', 'RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9', 'daily_return', 'volatility_20d', 'price_to_sma_50', 'price_to_sma_200', 'yield_spread_1mo_change', 'fed_funds_3mo_change', 'inflation_trend_3mo', 'fed_funds_6mo_lag', 'target_5d_up', 'yield_10y_change', 'yield_2y_change', 'yield_spread_change']
